# Real ONT Training Run

This notebook performs a full end-to-end run on a real Oxford Nanopore dataset for *E. coli* UTI89:

1. verify or download the raw reads and reference,
2. align a notebook-sized subset of real long reads with `mappy`,
3. build overlap-conditioned JSONL windows for OMEGA,
4. train the model for one epoch, and
5. evaluate on a held-out test split.

The run is intentionally sized to complete in a notebook while still using real sequencing data.


In [33]:
from __future__ import annotations

import copy
import gzip
import hashlib
import json
import os
import subprocess
import sys
from pathlib import Path
from urllib.request import urlretrieve

import mappy as mp
import pandas as pd
import pysam
import yaml

ROOT = Path('/Users/shanejayasundera/LRS-Error-Correction')
RAW_DIR = ROOT / 'data' / 'raw' / 'uti89'
RUN_DIR = ROOT / 'data' / 'real_uti89_notebook'
CKPT_DIR = ROOT / 'checkpoints' / 'real_uti89_notebook'
OUT_DIR = ROOT / 'outputs' / 'real_uti89_notebook'
NOTEBOOK_CONFIG = ROOT / 'configs' / 'real_uti89_notebook.yaml'

RAW_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

REFERENCE_GZ = RAW_DIR / 'reference.fna.gz'
REFERENCE_FA = RAW_DIR / 'reference.fna'
FASTQ_1 = RAW_DIR / 'ERR908493_1.fastq.gz'
FASTQ_2 = RAW_DIR / 'ERR908493_2.fastq.gz'
SORTED_BAM = RUN_DIR / 'aligned.sorted.bam'
MANIFEST = RUN_DIR / 'manifest.json'

print('python', sys.executable)
print('cwd', ROOT)
print('raw dir', RAW_DIR)


python /Users/shanejayasundera/anaconda3/envs/lrs_err_correct_env/bin/python
cwd /Users/shanejayasundera/LRS-Error-Correction
raw dir /Users/shanejayasundera/LRS-Error-Correction/data/raw/uti89


In [34]:
DOWNLOADS = {
    REFERENCE_GZ: 'https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/000/013/265/GCA_000013265.1_ASM1326v1/GCA_000013265.1_ASM1326v1_genomic.fna.gz',
    FASTQ_1: 'https://ftp.sra.ebi.ac.uk/vol1/fastq/ERR908/ERR908493/ERR908493_1.fastq.gz',
    FASTQ_2: 'https://ftp.sra.ebi.ac.uk/vol1/fastq/ERR908/ERR908493/ERR908493_2.fastq.gz',
}

for path, url in DOWNLOADS.items():
    if path.exists() and path.stat().st_size > 0:
        print('exists', path.name, path.stat().st_size)
    else:
        print('downloading', path.name)
        urlretrieve(url, path)
        print('downloaded', path.name, path.stat().st_size)

if (not REFERENCE_FA.exists()) or REFERENCE_FA.stat().st_size == 0:
    with gzip.open(REFERENCE_GZ, 'rb') as src, open(REFERENCE_FA, 'wb') as dst:
        dst.write(src.read())
    print('wrote', REFERENCE_FA)
else:
    print('exists', REFERENCE_FA)

if not (REFERENCE_FA.with_suffix(REFERENCE_FA.suffix + '.fai')).exists():
    pysam.faidx(str(REFERENCE_FA))
    print('indexed reference')
else:
    print('reference index exists')


exists reference.fna.gz 1540635
exists ERR908493_1.fastq.gz 9018804
exists ERR908493_2.fastq.gz 10102609
exists /Users/shanejayasundera/LRS-Error-Correction/data/raw/uti89/reference.fna
reference index exists


In [35]:
sys.path.insert(0, str(ROOT / 'src'))

from omega_longread.preprocessing import VariantLookup, build_read_encoding, build_window_example, iter_windows

MAX_READS = 500
MIN_READ_LENGTH = 1500
WINDOW_SIZE = 1024
WINDOW_OVERLAP = 256
MAX_SUPPORTS = 8
MIN_MAPQ = 5
MAX_INSERTIONS_PER_POS = 2

UNSORTED_BAM = RUN_DIR / 'aligned.unsorted.bam'
TRAIN_JSONL = RUN_DIR / 'train.jsonl'
VAL_JSONL = RUN_DIR / 'val.jsonl'
TEST_JSONL = RUN_DIR / 'test.jsonl'


def revcomp(seq: str) -> str:
    table = str.maketrans('ACGTNacgtn', 'TGCANtgcan')
    return seq.translate(table)[::-1]


def iter_fastq(path: Path):
    with gzip.open(path, 'rt') as f:
        while True:
            name = f.readline().strip()
            if not name:
                return
            seq = f.readline().strip()
            f.readline()
            qual = f.readline().strip()
            yield name[1:], seq, qual


def full_cigar(hit, query_len: int) -> str:
    if hit.strand < 0:
        left_soft = query_len - hit.q_en
        right_soft = hit.q_st
    else:
        left_soft = hit.q_st
        right_soft = query_len - hit.q_en
    parts = []
    if left_soft > 0:
        parts.append(f'{left_soft}S')
    parts.append(hit.cigar_str)
    if right_soft > 0:
        parts.append(f'{right_soft}S')
    return ''.join(parts)

reads = []
for fq in [FASTQ_1, FASTQ_2]:
    for name, seq, qual in iter_fastq(fq):
        if len(seq) >= MIN_READ_LENGTH:
            reads.append((name, seq, qual, fq.name))
reads.sort(key=lambda x: len(x[1]), reverse=True)
reads = reads[:MAX_READS]
print('selected reads', len(reads))

aligner = mp.Aligner(str(REFERENCE_FA), preset='map-ont')
fasta = pysam.FastaFile(str(REFERENCE_FA))
header = {
    'HD': {'VN': '1.6', 'SO': 'unsorted'},
    'SQ': [{'SN': name, 'LN': length} for name, length in zip(fasta.references, fasta.lengths)],
}
ref_to_id = {name: idx for idx, name in enumerate(fasta.references)}

written = 0
with pysam.AlignmentFile(str(UNSORTED_BAM), 'wb', header=header) as bam:
    for name, seq, qual, source in reads:
        hits = [h for h in aligner.map(seq) if h.is_primary]
        if not hits:
            continue
        hit = max(hits, key=lambda h: (h.mapq, h.mlen, h.blen))
        aln = pysam.AlignedSegment()
        aln.query_name = f'{source}|{name}'
        aln.flag = 16 if hit.strand < 0 else 0
        aln.reference_id = ref_to_id[hit.ctg]
        aln.reference_start = hit.r_st
        aln.mapping_quality = int(hit.mapq)
        aln.cigarstring = full_cigar(hit, len(seq))
        if hit.strand < 0:
            aln.query_sequence = revcomp(seq)
            aln.query_qualities = pysam.qualitystring_to_array(qual[::-1])
        else:
            aln.query_sequence = seq
            aln.query_qualities = pysam.qualitystring_to_array(qual)
        aln.set_tag('NM', int(hit.NM), value_type='i')
        bam.write(aln)
        written += 1
print('written alignments', written)

pysam.sort('-o', str(SORTED_BAM), str(UNSORTED_BAM))
pysam.index(str(SORTED_BAM))
print('sorted bam', SORTED_BAM)

variant_lookup = VariantLookup(None)
counts = {'train': 0, 'val': 0, 'test': 0}
files = {
    'train': open(TRAIN_JSONL, 'w', encoding='utf-8'),
    'val': open(VAL_JSONL, 'w', encoding='utf-8'),
    'test': open(TEST_JSONL, 'w', encoding='utf-8'),
}
try:
    with pysam.AlignmentFile(str(SORTED_BAM), 'rb', reference_filename=str(REFERENCE_FA)) as bam:
        for aln in bam.fetch(until_eof=True):
            enc = build_read_encoding(aln, fasta, max_insertions_per_pos=MAX_INSERTIONS_PER_POS)
            if enc is None or len(enc.target_bases) < 512:
                continue
            digest = int(hashlib.md5(enc.read_id.encode()).hexdigest(), 16) % 10
            split = 'train' if digest < 8 else ('val' if digest == 8 else 'test')
            for ws, we in iter_windows(len(enc.target_bases), WINDOW_SIZE, WINDOW_OVERLAP, 256):
                ex = build_window_example(
                    target_aln=aln,
                    read_encoding=enc,
                    bam=bam,
                    variant_lookup=variant_lookup,
                    confidence_lookup=None,
                    window_start=ws,
                    window_end=we,
                    max_supports=MAX_SUPPORTS,
                    min_mapq=MIN_MAPQ,
                    min_confident_fraction=0.0,
                    min_mapped_fraction=0.6,
                    support_disagreement_threshold=0.7,
                    min_support_depth=2,
                    max_insertions_per_pos=MAX_INSERTIONS_PER_POS,
                )
                if ex is None:
                    continue
                files[split].write(json.dumps(ex) + '\n')
                counts[split] += 1
finally:
    for handle in files.values():
        handle.close()
    fasta.close()

manifest = {
    'selected_reads': len(reads),
    'written_alignments': written,
    'window_counts': counts,
    'train_path': str(TRAIN_JSONL),
    'val_path': str(VAL_JSONL),
    'test_path': str(TEST_JSONL),
    'bam_path': str(SORTED_BAM),
}
MANIFEST.write_text(json.dumps(manifest, indent=2))
manifest


selected reads 500
written alignments 243
sorted bam /Users/shanejayasundera/LRS-Error-Correction/data/real_uti89_notebook/aligned.sorted.bam


{'selected_reads': 500,
 'written_alignments': 243,
 'window_counts': {'train': 712, 'val': 85, 'test': 74},
 'train_path': '/Users/shanejayasundera/LRS-Error-Correction/data/real_uti89_notebook/train.jsonl',
 'val_path': '/Users/shanejayasundera/LRS-Error-Correction/data/real_uti89_notebook/val.jsonl',
 'test_path': '/Users/shanejayasundera/LRS-Error-Correction/data/real_uti89_notebook/test.jsonl',
 'bam_path': '/Users/shanejayasundera/LRS-Error-Correction/data/real_uti89_notebook/aligned.sorted.bam'}

In [36]:
from collections import Counter
import json
from pathlib import Path

import pandas as pd

ROOT = Path('/Users/shanejayasundera/LRS-Error-Correction')
RUN_DIR = ROOT / 'data' / 'real_uti89_notebook'
TRAIN_JSONL = RUN_DIR / 'train.jsonl'
VAL_JSONL = RUN_DIR / 'val.jsonl'
TEST_JSONL = RUN_DIR / 'test.jsonl'

ID_TO_EDIT = {
    0: 'COPY',
    1: 'SUB_A',
    2: 'SUB_C',
    3: 'SUB_G',
    4: 'SUB_T',
    5: 'DEL',
    6: 'INS_A',
    7: 'INS_C',
    8: 'INS_G',
    9: 'INS_T',
    10: 'PAD',
}


def count_edit_labels(path: Path):
    all_tokens = Counter()
    core_tokens = Counter()
    insertion_tokens = Counter()
    windows = 0
    positions = 0

    with path.open('r', encoding='utf-8') as handle:
        for line in handle:
            item = json.loads(line)
            windows += 1
            for label_slots in item['edit_labels']:
                positions += 1
                for slot_idx, token_id in enumerate(label_slots):
                    token = ID_TO_EDIT.get(token_id, str(token_id))
                    all_tokens[token] += 1
                    if slot_idx == len(label_slots) - 1:
                        core_tokens[token] += 1
                    elif token != 'PAD':
                        insertion_tokens[token] += 1

    def to_df(counter: Counter, split: str) -> pd.DataFrame:
        total = sum(counter.values()) or 1
        rows = [
            {
                'split': split,
                'token': token,
                'count': count,
                'fraction': count / total,
            }
            for token, count in sorted(counter.items(), key=lambda kv: (-kv[1], kv[0]))
        ]
        return pd.DataFrame(rows)

    summary = pd.DataFrame([
        {
            'split': path.stem,
            'windows': windows,
            'positions': positions,
            'all_token_total': sum(all_tokens.values()),
            'core_token_total': sum(core_tokens.values()),
            'non_pad_insertion_total': sum(insertion_tokens.values()),
        }
    ])
    return summary, to_df(all_tokens, path.stem), to_df(core_tokens, path.stem), to_df(insertion_tokens, path.stem)


split_paths = [TRAIN_JSONL, VAL_JSONL, TEST_JSONL]
summary_frames = []
all_frames = []
core_frames = []
ins_frames = []

for split_path in split_paths:
    if not split_path.exists():
        raise FileNotFoundError(f'Missing dataset split: {split_path}')
    summary_df, all_df, core_df, ins_df = count_edit_labels(split_path)
    summary_frames.append(summary_df)
    all_frames.append(all_df)
    core_frames.append(core_df)
    ins_frames.append(ins_df)

label_summary_df = pd.concat(summary_frames, ignore_index=True)
all_label_counts_df = pd.concat(all_frames, ignore_index=True)
core_label_counts_df = pd.concat(core_frames, ignore_index=True)
insertion_label_counts_df = pd.concat(ins_frames, ignore_index=True)

print('Dataset summary')
print(label_summary_df.to_string(index=False))
print()
print('Core-slot label counts')
print(core_label_counts_df.to_string(index=False))
print()
print('Insertion-slot label counts (PAD removed)')
print(insertion_label_counts_df.to_string(index=False))


Dataset summary
split  windows  positions  all_token_total  core_token_total  non_pad_insertion_total
train      712     697619          2092857            697619                    90553
  val       85      84475           253425             84475                    10361
 test       74      73708           221124             73708                     9321

Core-slot label counts
split token  count  fraction
train  COPY 552617  0.792147
train   DEL  45656  0.065445
train SUB_C  27766  0.039801
train SUB_G  25549  0.036623
train SUB_T  23338  0.033454
train SUB_A  22693  0.032529
  val  COPY  65953  0.780740
  val   DEL   6165  0.072980
  val SUB_C   3332  0.039444
  val SUB_A   3220  0.038118
  val SUB_G   3169  0.037514
  val SUB_T   2636  0.031204
 test  COPY  58762  0.797227
 test   DEL   4724  0.064091
 test SUB_C   2800  0.037988
 test SUB_A   2732  0.037065
 test SUB_G   2560  0.034732
 test SUB_T   2130  0.028898

Insertion-slot label counts (PAD removed)
split token  count  fr

In [37]:
ABLATION_RUNS = [
    {
        'name': 'full',
        'support_mode': 'full',
        'description': 'Target + overlap-conditioned support encoder',
    },
    {
        'name': 'target_only',
        'support_mode': 'target_only',
        'description': 'Target encoder only; support path disabled',
    },
    {
        'name': 'masked_target',
        'support_mode': 'masked_target',
        'description': 'Support-conditioned with masked target features',
    },
]

ABLATION_CONFIG_DIR = ROOT / 'configs' / 'real_uti89_notebook_ablation'
ABLATION_CONFIG_DIR.mkdir(parents=True, exist_ok=True)
ABLATION_RESULTS = []

base_config = {
    'model': {
        'd_model': 32,
        'num_heads': 4,
        'num_layers': 1,
        'ff_mult': 2,
        'dropout': 0.1,
        'support_feature_dim': 8,
        'max_supports': 8,
        'max_insertions_per_pos': 2,
        'conv_kernel_size': 5,
        'support_mode': 'full',
    },
    'data': {
        'train_path': str(TRAIN_JSONL.relative_to(ROOT)),
        'val_path': str(VAL_JSONL.relative_to(ROOT)),
        'test_path': str(TEST_JSONL.relative_to(ROOT)),
        'max_target_len': 1024,
        'max_supports': 8,
        'max_support_len': 1024,
        'num_workers': 0,
    },
    'train': {
        'seed': 42,
        'batch_size': 4,
        'epochs': 5,
        'lr': 1e-4,
        'weight_decay': 0.01,
        'grad_clip_norm': 0.5,
        'device': 'cpu',
        'mixed_precision': False,
        'log_every': 20,
        'eval_every': 1,
        'save_dir': '',
        'checkpoint_metric': 'composite_sequence',
        'checkpoint_metric_mode': 'max',
        'checkpoint_overcorrection_weight': 0.5,
        'checkpoint_length_ratio_weight': 0.35,
    },
    'loss': {
        'lambda_edit': 1.0,
        'lambda_sequence': 0.5,
        'lambda_length': 0.3,
        'lambda_insertion_count': 0.2,
        'lambda_trust': 0.1,
        'lambda_hard_edit': 0.2,
        'lambda_support': 0.2,
        'lambda_preserve': 0.15,
        'lambda_uncertainty': 0.1,
        'homopolymer_weight_scale': 0.15,
        'label_smoothing': 0.0,
        'edit_class_weights': [],
        'auto_edit_class_weights': True,
        'edit_class_weight_power': 1.0,
        'edit_class_weight_count_smoothing': 1.0,
        'edit_class_weight_min': 0.25,
        'edit_class_weight_max': 8.0,
        'substitution_loss_scale': 1.75,
        'deletion_loss_scale': 1.25,
        'insertion_loss_scale': 1.1,
        'hard_edit_uncertainty_power': 1.0,
    },
}

for run in ABLATION_RUNS:
    run_name = run['name']
    cfg = copy.deepcopy(base_config)
    cfg['model']['support_mode'] = run['support_mode']
    cfg['train']['save_dir'] = str((ROOT / 'checkpoints' / 'real_uti89_notebook_ablation' / run_name).relative_to(ROOT))
    config_path = ABLATION_CONFIG_DIR / f'{run_name}.yaml'
    with open(config_path, 'w', encoding='utf-8') as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    run['config_path'] = config_path
    run['checkpoint_dir'] = ROOT / cfg['train']['save_dir']
    run['output_dir'] = ROOT / 'outputs' / 'real_uti89_notebook_ablation' / run_name
    run['output_dir'].mkdir(parents=True, exist_ok=True)

pd.DataFrame([
    {
        'run': run['name'],
        'support_mode': run['support_mode'],
        'config_path': str(run['config_path'].relative_to(ROOT)),
        'checkpoint_dir': str(run['checkpoint_dir'].relative_to(ROOT)),
        'output_dir': str(run['output_dir'].relative_to(ROOT)),
    }
    for run in ABLATION_RUNS
])


,run,support_mode,config_path,checkpoint_dir,output_dir
0,full,full,configs/real_uti89_notebook_ablation/full.yaml,checkpoints/real_uti89_notebook_ablation/full,outputs/real_uti89_notebook_ablation/full
1,target_only,target_only,configs/real_uti89_notebook_ablation/target_on...,checkpoints/real_uti89_notebook_ablation/targe...,outputs/real_uti89_notebook_ablation/target_only
2,masked_target,masked_target,configs/real_uti89_notebook_ablation/masked_ta...,checkpoints/real_uti89_notebook_ablation/maske...,outputs/real_uti89_notebook_ablation/masked_ta...


In [ ]:
env = os.environ.copy()
env['PYTHONPATH'] = str(ROOT / 'src')
env['PYTHONUNBUFFERED'] = '1'

for run in ABLATION_RUNS:
    train_cmd = [
        sys.executable,
        'scripts/train.py',
        '--config',
        str(run['config_path']),
    ]
    print(f"\n=== training {run['name']} ({run['description']}) ===")
    print('running', ' '.join(train_cmd))
    subprocess.run(train_cmd, cwd=ROOT, env=env, check=True)



=== training full (Target + overlap-conditioned support encoder) ===
running /Users/shanejayasundera/anaconda3/envs/lrs_err_correct_env/bin/python scripts/train.py --config /Users/shanejayasundera/LRS-Error-Correction/configs/real_uti89_notebook_ablation/full.yaml


/Users/shanejayasundera/LRS-Error-Correction/scripts/train.py:192: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=cfg.train.mixed_precision and device.type == "cuda")


resolved edit class weights:
[
  {
    "token_id": 0,
    "token": "COPY",
    "weight": 0.25
  },
  {
    "token_id": 1,
    "token": "SUB_A",
    "weight": 3.4731674194335938
  },
  {
    "token_id": 2,
    "token": "SUB_C",
    "weight": 2.838623523712158
  },
  {
    "token_id": 3,
    "token": "SUB_G",
    "weight": 3.0849337577819824
  },
  {
    "token_id": 4,
    "token": "SUB_T",
    "weight": 3.3771824836730957
  },
  {
    "token_id": 5,
    "token": "DEL",
    "weight": 1.7263520956039429
  },
  {
    "token_id": 6,
    "token": "INS_A",
    "weight": 3.618586778640747
  },
  {
    "token_id": 7,
    "token": "INS_C",
    "weight": 3.355044364929199
  },
  {
    "token_id": 8,
    "token": "INS_G",
    "weight": 3.3116281032562256
  },
  {
    "token_id": 9,
    "token": "INS_T",
    "weight": 3.6692919731140137
  },
  {
    "token_id": 10,
    "token": "PAD",
    "weight": 0.0
  }
]
epoch 1/5


/Users/shanejayasundera/anaconda3/envs/lrs_err_correct_env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/shanejayasundera/LRS-Error-Correction/scripts/train.py:87: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=cfg.train.mixed_precision):


train step=20: {"loss": 5.419593715667725, "edit_loss": 3.9559518456459046, "sequence_loss": 1.401836371421814, "length_loss": 0.2296796441078186, "insertion_count_loss": 0.621444919705391, "trust_loss": 0.6965252429246902, "hard_edit_loss": 0.11852907706052065, "selective_hard_edit_loss": 2.2813924074172975, "support_loss": 1.347601342201233, "preserve_loss": 0.6271654158830643, "uncertainty_loss": 1.1257729172706603, "predicted_length": 1227.9117248535156, "target_length": 1037.55, "predicted_insertions": 743.643179321289, "target_insertions": 127.4375, "trust_gate_mean": 0.517487421631813, "support_confidence_mean": 0.5134596660733223, "support_uncertainty_mean": 0.2556370906531811, "selective_hard_edit_fraction": 0.04739990234375, "edit_accuracy": 0.13334150277078152, "core_edit_accuracy": 0.1427445612847805, "copy_recall": 0.14774658009409905, "copy_precision": 0.8016782373189926, "delete_recall": 0.06111345924437046, "delete_precision": 0.06197807639837265, "sub_recall": 0.516729

...

resolved edit class weights:
[
  {
    "token_id": 0,
    "token": "COPY",
    "weight": 0.25
  },
  {
    "token_id": 1,
    "token": "SUB_A",
    "weight": 3.4731674194335938
  },
  {
    "token_id": 2,
    "token": "SUB_C",
    "weight": 2.838623523712158
  },
  {
    "token_id": 3,
    "token": "SUB_G",
    "weight": 3.0849337577819824
  },
  {
    "token_id": 4,
    "token": "SUB_T",
    "weight": 3.3771824836730957
  },
  {
    "token_id": 5,
    "token": "DEL",
    "weight": 1.7263520956039429
  },
  {
    "token_id": 6,
    "token": "INS_A",
    "weight": 3.618586778640747
  },
  {
    "token_id": 7,
    "token": "INS_C",
    "weight": 3.355044364929199
  },
  {
    "token_id": 8,
    "token": "INS_G",
    "weight": 3.3116281032562256
  },
  {
    "token_id": 9,
    "token": "INS_T",
    "weight": 3.6692919731140137
  },
  {
    "token_id": 10,
    "token": "PAD",
    "weight": 0.0
  }
]
epoch 1/5


/Users/shanejayasundera/anaconda3/envs/lrs_err_correct_env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/shanejayasundera/LRS-Error-Correction/scripts/train.py:87: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=cfg.train.mixed_precision):


train step=20: {"loss": 6.094512629508972, "edit_loss": 3.98977677822113, "sequence_loss": 1.3982456922531128, "length_loss": 0.221267107501626, "insertion_count_loss": 0.624784055352211, "trust_loss": 7.093707847595215, "hard_edit_loss": 0.11427805032581091, "selective_hard_edit_loss": 2.315255343914032, "support_loss": 1.3428927838802338, "preserve_loss": 0.6258541613817215, "uncertainty_loss": 1.1959290742874145, "predicted_length": 1218.8971069335937, "target_length": 1037.55, "predicted_insertions": 746.0983093261718, "target_insertions": 127.4375, "trust_gate_mean": 0.0, "support_confidence_mean": 0.5134596660733223, "support_uncertainty_mean": 0.2556370906531811, "selective_hard_edit_fraction": 0.04739990234375, "edit_accuracy": 0.1775765113532543, "core_edit_accuracy": 0.19446261078119279, "copy_recall": 0.2175498865544796, "copy_precision": 0.7975010484457016, "delete_recall": 0.04508294216357171, "delete_precision": 0.06283615990541876, "sub_recall": 0.45864326059818267, "sub

/Users/shanejayasundera/LRS-Error-Correction/scripts/train.py:141: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=cfg.train.mixed_precision):


{
  "train_loss": 5.844933598229055,
  "train_edit_loss": 3.731839279110512,
  "train_sequence_loss": 1.4073528352748141,
  "train_length_loss": 0.3181185841644078,
  "train_insertion_count_loss": 0.6369876597034797,
  "train_trust_loss": 7.080925322650524,
  "train_hard_edit_loss": 0.13551162763006902,
  "train_selective_hard_edit_loss": 2.188569603341349,
  "train_support_loss": 1.3584883219740365,
  "train_preserve_loss": 0.5534897724564156,
  "train_uncertainty_loss": 0.9666878711641504,
  "train_predicted_length": 1320.4185413832074,
  "train_target_length": 1042.8595505617977,
  "train_predicted_insertions": 757.9026612699702,
  "train_target_insertions": 127.1811797752809,
  "train_trust_gate_mean": 0.0,
  "train_support_confidence_mean": 0.5125344332014576,
  "train_support_uncertainty_mean": 0.25419968722409075,
  "train_selective_hard_edit_fraction": 0.05200722077442856,
  "train_edit_accuracy": 0.1356623883076598,
  "train_core_edit_accuracy": 0.14306245568428147,
  "train_c

In [ ]:
for run in ABLATION_RUNS:
    test_summary = run['output_dir'] / 'test_summary.json'
    test_predictions = run['output_dir'] / 'test_predictions.jsonl'
    test_cmd = [
        sys.executable,
        'scripts/test.py',
        '--config',
        str(run['config_path']),
        '--checkpoint',
        str(run['checkpoint_dir'] / 'best_model.pt'),
        '--summary-out',
        str(test_summary),
        '--predictions-out',
        str(test_predictions),
    ]
    print(f"\n=== testing {run['name']} ===")
    print('running', ' '.join(test_cmd))
    subprocess.run(test_cmd, cwd=ROOT, env=env, check=True)
    run['test_summary_path'] = test_summary
    run['test_predictions_path'] = test_predictions



=== testing full ===
running /Users/shanejayasundera/anaconda3/envs/lrs_err_correct_env/bin/python scripts/test.py --config /Users/shanejayasundera/LRS-Error-Correction/configs/real_uti89_notebook_ablation/full.yaml --checkpoint /Users/shanejayasundera/LRS-Error-Correction/checkpoints/real_uti89_notebook_ablation/full/best_model.pt --summary-out /Users/shanejayasundera/LRS-Error-Correction/outputs/real_uti89_notebook_ablation/full/test_summary.json --predictions-out /Users/shanejayasundera/LRS-Error-Correction/outputs/real_uti89_notebook_ablation/full/test_predictions.jsonl


/Users/shanejayasundera/anaconda3/envs/lrs_err_correct_env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/shanejayasundera/LRS-Error-Correction/scripts/test.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=cfg.train.mixed_precision and device.type == "cuda"):
/Users/shanejayasundera/LRS-Error-Correction/scripts/test.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=cfg.train.mixed_precision and device.type == "cuda"):


{
  "loss": 4.592140900461297,
  "edit_loss": 3.1962926889720715,
  "sequence_loss": 1.402444513220536,
  "length_loss": 0.40712853168186386,
  "insertion_count_loss": 0.5162420962986193,
  "trust_loss": 0.6557428868193376,
  "hard_edit_loss": 0.14680387903200953,
  "support_loss": 1.3884824263422113,
  "preserve_loss": 0.3334433154055947,
  "uncertainty_loss": 0.4659097241727929,
  "predicted_length": 1465.4095201994244,
  "target_length": 1057.1842105263158,
  "predicted_insertions": 660.4976806640625,
  "target_insertions": 125.59210526315789,
  "trust_gate_mean": 0.5414075537731773,
  "support_confidence_mean": 0.6429336886656912,
  "support_uncertainty_mean": 0.19815600663423538,
  "edit_accuracy": 0.6774928412939373,
  "core_edit_accuracy": 0.7631618662884361,
  "copy_recall": 0.9510802250159415,
  "copy_precision": 0.801795808892501,
  "delete_recall": 0.01363163019873594,
  "delete_precision": 0.10970341571067509,
  "sub_recall": 0.05747603889750807,
  "sub_precision": 0.173597

/Users/shanejayasundera/anaconda3/envs/lrs_err_correct_env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/shanejayasundera/LRS-Error-Correction/scripts/test.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=cfg.train.mixed_precision and device.type == "cuda"):
/Users/shanejayasundera/LRS-Error-Correction/scripts/test.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=cfg.train.mixed_precision and device.type == "cuda"):


{
  "loss": 5.459262270676462,
  "edit_loss": 3.2377412570150277,
  "sequence_loss": 1.4055874347686768,
  "length_loss": 0.40790971486192,
  "insertion_count_loss": 0.5437748808609811,
  "trust_loss": 8.882457457090679,
  "hard_edit_loss": 0.14315130051813627,
  "support_loss": 1.3451546242362575,
  "preserve_loss": 0.33725446148922567,
  "uncertainty_loss": 0.5110446246046769,
  "predicted_length": 1466.7071019222863,
  "target_length": 1057.1842105263158,
  "predicted_insertions": 687.6172645970395,
  "target_insertions": 125.59210526315789,
  "trust_gate_mean": 0.0,
  "support_confidence_mean": 0.6429336886656912,
  "support_uncertainty_mean": 0.19815600663423538,
  "edit_accuracy": 0.7056251764297485,
  "core_edit_accuracy": 0.7947856564270822,
  "copy_recall": 0.9952790454814309,
  "copy_precision": 0.7984265340001959,
  "delete_recall": 0.0,
  "delete_precision": 0.0,
  "sub_recall": 0.00621457309707215,
  "sub_precision": 0.18378516953242452,
  "ins_recall": 0.00547963415125482

/Users/shanejayasundera/anaconda3/envs/lrs_err_correct_env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/shanejayasundera/LRS-Error-Correction/scripts/test.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=cfg.train.mixed_precision and device.type == "cuda"):
/Users/shanejayasundera/LRS-Error-Correction/scripts/test.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=cfg.train.mixed_precision and device.type == "cuda"):


{
  "loss": 4.657378748843544,
  "edit_loss": 3.274605964359484,
  "sequence_loss": 1.3973982961554277,
  "length_loss": 0.39639709184044286,
  "insertion_count_loss": 0.5257629642361089,
  "trust_loss": 0.6563156247138977,
  "hard_edit_loss": 0.1410677260474155,
  "support_loss": 1.325049218378569,
  "preserve_loss": 0.33577785366459895,
  "uncertainty_loss": 0.5078017633212241,
  "predicted_length": 1454.189119037829,
  "target_length": 1057.1842105263158,
  "predicted_insertions": 670.6429860968339,
  "target_insertions": 125.59210526315789,
  "trust_gate_mean": 0.537244765382064,
  "support_confidence_mean": 0.6429336886656912,
  "support_uncertainty_mean": 0.19815600663423538,
  "edit_accuracy": 0.7085588448926022,
  "core_edit_accuracy": 0.7982912847870275,
  "copy_recall": 1.0,
  "copy_precision": 0.7982912847870275,
  "delete_recall": 0.0,
  "delete_precision": 0.0,
  "sub_recall": 0.0,
  "sub_precision": 0.0,
  "ins_recall": 0.0,
  "ins_precision": 0.0,
  "sequence_edit_distan

In [ ]:
comparison_rows = []
history_frames = []

for run in ABLATION_RUNS:
    history_path = run['checkpoint_dir'] / 'history.json'
    test_summary_path = run['test_summary_path']
    history = json.loads(history_path.read_text())
    test_metrics = json.loads(test_summary_path.read_text())
    history_df = pd.DataFrame(history)
    history_df.insert(0, 'run', run['name'])
    history_frames.append(history_df)

    comparison_rows.append({
        'run': run['name'],
        'support_mode': run['support_mode'],
        'description': run['description'],
        'best_epoch': int(history_df.iloc[history_df['val_selection_score'].idxmax()]['epoch']),
        'val_selection_score_max': float(history_df['val_selection_score'].max()),
        'final_val_sequence_identity': float(history_df.iloc[-1]['val_sequence_identity']),
        'final_val_length_ratio': float(history_df.iloc[-1]['val_predicted_length_ratio']),
        'final_val_overcorrection_rate': float(history_df.iloc[-1]['val_overcorrection_rate']),
        'test_sequence_identity': float(test_metrics['sequence_identity']),
        'test_sequence_normalized_edit_distance': float(test_metrics['sequence_normalized_edit_distance']),
        'test_predicted_length_ratio': float(test_metrics['predicted_length_ratio']),
        'test_predicted_insertions_per_window': float(test_metrics['predicted_insertions_per_window']),
        'test_predicted_insertion_burst_length': float(test_metrics['predicted_insertion_burst_length']),
        'test_sub_recall': float(test_metrics['sub_recall']),
        'test_delete_recall': float(test_metrics['delete_recall']),
        'test_overcorrection_rate': float(test_metrics['overcorrection_rate']),
    })

ablation_history_df = pd.concat(history_frames, ignore_index=True)
ablation_summary_df = pd.DataFrame(comparison_rows).sort_values(
    by=['test_sequence_identity', 'test_sequence_normalized_edit_distance'],
    ascending=[False, True],
).reset_index(drop=True)

ablation_summary_path = ROOT / 'outputs' / 'real_uti89_notebook_ablation' / 'ablation_summary.json'
ablation_history_path = ROOT / 'outputs' / 'real_uti89_notebook_ablation' / 'ablation_history.json'
with open(ablation_summary_path, 'w', encoding='utf-8') as f:
    json.dump(comparison_rows, f, indent=2)
with open(ablation_history_path, 'w', encoding='utf-8') as f:
    json.dump(ablation_history_df.to_dict(orient='records'), f, indent=2)

print(ablation_summary_path)
print(ablation_history_path)
display(ablation_summary_df)
display(ablation_history_df)


/Users/shanejayasundera/LRS-Error-Correction/outputs/real_uti89_notebook_ablation/ablation_summary.json
/Users/shanejayasundera/LRS-Error-Correction/outputs/real_uti89_notebook_ablation/ablation_history.json


,run,support_mode,description,best_epoch,val_selection_score_max,final_val_sequence_identity,final_val_length_ratio,final_val_overcorrection_rate,test_sequence_identity,test_sequence_normalized_edit_distance,test_predicted_length_ratio,test_predicted_insertions_per_window,test_predicted_insertion_burst_length,test_sub_recall,test_delete_recall,test_overcorrection_rate
0,masked_target,masked_target,Support-conditioned with masked target features,2,0.664477,0.680598,0.953407,0.000000,0.695578,0.304422,0.941315,0.000000,0.000000,0.000000,0.000000,0.000000
1,target_only,target_only,Target encoder only; support path disabled,2,0.660899,0.593651,0.997660,0.113234,0.689604,0.310396,0.949244,8.710526,1.067216,0.006215,0.000000,0.003518
2,full,full,Target + overlap-conditioned support encoder,2,0.628481,0.573980,1.101380,0.164411,0.672762,0.327238,0.943606,10.000000,1.031784,0.057476,0.013632,0.024852


,run,train_loss,train_edit_loss,train_sequence_loss,train_length_loss,train_insertion_count_loss,train_trust_loss,train_hard_edit_loss,train_support_loss,train_preserve_loss,...,val_high_support_entropy_sequence_identity,val_support_entropy_window_mean,val_trust_gate_window_mean,val_trust_gate_low_entropy_mean,val_trust_gate_high_entropy_mean,val_trust_gate_homopolymer_mean,val_trust_gate_non_homopolymer_mean,val_overcorrection_rate,epoch,val_selection_score
0,full,5.117469,3.662355,1.410455,0.330221,0.609846,0.692919,0.147172,1.365082,0.554751,...,0.0,0.012729,0.552857,0.552904,0.0,0.559702,0.552427,0.229540,1,0.359535
1,full,4.783243,3.334486,1.406983,0.445382,0.553984,0.685579,0.181466,1.400276,0.514453,...,0.0,0.012729,0.537797,0.537825,0.0,0.545542,0.537355,0.027474,2,0.628481
2,full,4.720803,3.254669,1.407743,0.458822,0.540799,0.675399,0.194617,1.458811,0.546220,...,0.0,0.012729,0.504262,0.504224,0.0,0.517709,0.503505,0.105943,3,0.507063
3,full,4.664693,3.203897,1.404652,0.447379,0.531147,0.659840,0.186198,1.528794,0.489812,...,0.0,0.012729,0.497307,0.497193,0.0,0.519271,0.496094,0.144272,4,0.477398
4,full,4.629358,3.169989,1.398115,0.436784,0.528015,0.643411,0.188888,1.562393,0.494650,...,0.0,0.012729,0.483552,0.483385,0.0,0.519025,0.481626,0.164411,5,0.456291
5,target_only,5.844934,3.731839,1.407353,0.318119,0.636988,7.080925,0.135512,1.358488,0.553490,...,0.0,0.012729,0.000000,0.000000,0.0,0.000000,0.000000,0.251406,1,0.273511
6,target_only,5.498404,3.383794,1.411088,0.445858,0.601317,7.176306,0.171327,1.370525,0.520329,...,0.0,0.012729,0.000000,0.000000,0.0,0.000000,0.000000,0.000920,2,0.660899
7,target_only,5.376616,3.292098,1.409856,0.462962,0.564029,6.892560,0.193116,1.391639,0.551261,...,0.0,0.012729,0.000000,0.000000,0.0,0.000000,0.000000,0.040593,3,0.607366
8,target_only,5.360665,3.243884,1.413297,0.468642,0.555424,7.209894,0.188173,1.441121,0.498888,...,0.0,0.012729,0.000000,0.000000,0.0,0.000000,0.000000,0.089432,4,0.559713
9,target_only,5.319789,3.202494,1.412316,0.473962,0.557348,7.078586,0.191304,1.498570,0.506180,...,0.0,0.012729,0.000000,0.000000,0.0,0.000000,0.000000,0.113234,5,0.536214
